# Study 933 — Same Issuer, Two Ladders — the teardown

The pairwise same-issuer estimator, the Newey-West *t*, joint block-bootstrap CIs, the per-issuer table and its Jensen trap, the era cut, the cost and borrow sweeps, the stress windows, the long-history extension, the survivorship probe, the PFF fallback and the live synthetic control. Every real number is frozen from `docs/results.md` (Fingerprint `9ae51d88019e`).

**Design.** Two equal-weight ladders over 8 issuer pairs, monthly rebalance, weights formed through day *t* and applied at *t+1* (the study's only execution lag), 25 bps one-way x NAV on turnover (a PROXY), both series taken excess-of-cash (BIL total return). Daily **total-return** closes throughout — price-only is meaningless between two coupon instruments.

In [1]:
R = {'start': '2022-02-02', 'end': '2026-06-30', 'n_days': 1105, 'fp': '9ae51d88019e', 'n_pairs': 8, 'cost_bps': 25, 'turnover': 75, 'pref_sharpe': 0.368, 'pref_cagr': 5.35, 'pref_vol': 18.96, 'pref_dd': -30.82, 'pref_t': 0.68, 'bond_sharpe': 0.202, 'bond_cagr': 1.82, 'bond_vol': 13.31, 'bond_dd': -20.4, 'bond_t': 0.38, 'adv': 0.166, 't_ladder': 0.65, 'ret_spread': 4.29, 'pairwise': 3.1, 't_pairwise': 0.55, 'vol_ratio': 1.42, 'ci_pref_lo': -0.736, 'ci_pref_hi': 1.351, 'ci_bond_lo': -0.823, 'ci_bond_hi': 1.256, 'ci_adv_lo': -0.47, 'ci_adv_hi': 0.804, 'ci_adv_neg': 32.3, 'ci_pw_lo': -6.8, 'ci_pw_hi': 13.57, 'wins': 6, 'n_pairs_win': 8, 'wilson_lo': 0.41, 'wilson_hi': 0.93, 'median_pair': 0.7, 'riskier': 6, 'ex_rily': 0.81, 't_ex_rily': 0.17, 'rily_spread': 19.15, 'rily_pref_cagr': -7.71, 'rily_bond_cagr': 2.58, 'era_e_n': 542, 'era_e_adv': -0.16, 'era_e_pw': -1.16, 'era_e_t': -0.21, 'era_l_n': 564, 'era_l_adv': 0.1, 'era_l_pw': 7.17, 'era_l_t': 0.72, 'cost5_adv': 0.166, 'cost5_spread': 4.33, 'cost100_adv': 0.167, 'cost100_spread': 4.14, 'ls_gross': 4.29, 'ls_sharpe0': 0.282, 'ls_t': 0.65, 'ls_300': 1.29, 'ls_sharpe300': 0.085, 'ls_dd': -25.9, 'st_2022': 0.06, 'st_banks': 2.89, 'st_2024': 5.28, 'st_tariff': -1.03, 'st_covid': 9.53, 'covid_pref': -10.25, 'covid_bond': -19.78, 'long_n': 1821, 'long_pw': -0.43, 'long_t': -0.14, 'long_adv': -0.064, 'long_volratio': 0.95, 'alt_pw': 3.84, 'alt_t': 0.68, 'alt_adv': 0.222, 'sach_pref': -17.4, 'sach_bond': 21.6, 'sach_spread': -10.63, 'sach_t': -0.85, 'pff_sharpe': -0.191, 'pff_cagr': -2.6, 'pff_gap': -4.75, 'pff_t': -0.9, 'pff_corr': 0.42, 'syn_pl': 5.74, 'syn_pl_fire': 20, 'syn_nl': -0.26, 'syn_nl_fire': 0, 'syn_planted': 6.0, 'stale_worst': 'CMS-PB', 'stale_worst_zero': 15.5, 'stale_worst_ac1': -0.28, 'stale_worst_vol_d': 24.0, 'stale_worst_vol_m': 11.4, 'stale_worst_bond_vol_m': 13.5, 'stale_pref_mean': 5.4, 'stale_bond_mean': 5.0, 'riskier_d': 6, 'riskier_w': 4, 'riskier_m': 2, 'ratio_d': 1.31, 'ratio_w': 1.3, 'ratio_m': 1.3, 'ex2_pairs': 6, 'ex2_pw': -0.6, 'ex2_t': -0.21, 'ex2_adv': -0.116, 'ex2_ratio_d': 0.94, 'ex2_ratio_m': 0.79, 'ex2_riskier_m': 0, 'ex2_dd_pref': -20.1, 'ex2_dd_bond': -19.3, 'ex1_ratio_m': 1.27, 'ex1_riskier_m': 1, 'ex1_pairs': 7, 'wide_step_pairs': 2, 'narrow_step_pairs': 6}

## 1. Why *pairwise*

A preferred-vs-bond race across *different* issuers confounds seniority with credit quality, sector and duration. Holding the obligor fixed makes the issuer's spread factor cancel in the difference:

$$ d_{i,t} = r^{pref}_{i,t} - r^{bond}_{i,t} = (\beta^{pref}_i - \beta^{bond}_i) f_{i,t} + \Delta\text{carry}_i + \varepsilon_{i,t} $$

so $\overline{d}$ estimates the paid seniority premium plus a residual beta term, not a cross-sectional credit tilt. The cost is $n$: only a handful of US issuers have ever had both rungs listed at once.

> 💡 *In plain words:* comparing two rungs of the same company removes the question 'is this a better company?' and leaves only 'is this a worse seat?'.

In [2]:
print(f"preferred ladder : exSharpe {R['pref_sharpe']:+.3f}  CAGR {R['pref_cagr']:+.2f}%  "
      f"vol {R['pref_vol']:.2f}%  DD {R['pref_dd']:.1f}%  HAC t {R['pref_t']:+.2f}")
print(f"baby-bond ladder : exSharpe {R['bond_sharpe']:+.3f}  CAGR {R['bond_cagr']:+.2f}%  "
      f"vol {R['bond_vol']:.2f}%  DD {R['bond_dd']:.1f}%  HAC t {R['bond_t']:+.2f}")
print()
print(f"Sharpe advantage (pref - bond) : {R['adv']:+.3f}")
print(f"ladder return spread          : {R['ret_spread']:+.2f}%/yr  (HAC t = {R['t_ladder']:+.2f})")
print(f"PAIRWISE same-issuer spread   : {R['pairwise']:+.2f}%/yr  (HAC t = {R['t_pairwise']:+.2f})")
print(f"vol ratio pref/bond {R['vol_ratio']:.2f}   annual ladder turnover {R['turnover']}%")

preferred ladder : exSharpe +0.368  CAGR +5.35%  vol 18.96%  DD -30.8%  HAC t +0.68
baby-bond ladder : exSharpe +0.202  CAGR +1.82%  vol 13.31%  DD -20.4%  HAC t +0.38

Sharpe advantage (pref - bond) : +0.166
ladder return spread          : +4.29%/yr  (HAC t = +0.65)
PAIRWISE same-issuer spread   : +3.10%/yr  (HAC t = +0.55)
vol ratio pref/bond 1.42   annual ladder turnover 75%


## 2. Bootstrap — the arms are resampled *jointly*

The two rungs of one balance sheet are strongly contemporaneously correlated; resampling them independently would badly overstate the precision of the difference. 2,000 circular block draws, 21-day blocks, same block index for both arms.

In [3]:
print(f"pref ladder exSharpe {R['pref_sharpe']:+.3f}  95% CI [{R['ci_pref_lo']:+.3f}, {R['ci_pref_hi']:+.3f}]")
print(f"bond ladder exSharpe {R['bond_sharpe']:+.3f}  95% CI [{R['ci_bond_lo']:+.3f}, {R['ci_bond_hi']:+.3f}]")
print(f"Sharpe ADVANTAGE     {R['adv']:+.3f}  95% CI [{R['ci_adv_lo']:+.3f}, {R['ci_adv_hi']:+.3f}]  "
      f"share<0 {R['ci_adv_neg']:.1f}%")
print(f"pairwise spread      {R['pairwise']:+.2f}%/yr  95% CI [{R['ci_pw_lo']:+.2f}%, {R['ci_pw_hi']:+.2f}%]")

pref ladder exSharpe +0.368  95% CI [-0.736, +1.351]
bond ladder exSharpe +0.202  95% CI [-0.823, +1.256]
Sharpe ADVANTAGE     +0.166  95% CI [-0.470, +0.804]  share<0 32.3%
pairwise spread      +3.10%/yr  95% CI [-6.80%, +13.57%]


## 3. The cross-section — and a Jensen trap

Wins (spread > 0): **6/8**, Wilson 95% **[0.41, 0.93]** — the hit rate cannot reject a coin. Median pair spread **+0.70%/yr** against a panel mean of **+3.10%**: the mean is not the typical pair.

**B. Riley** contributes an arithmetic **+19.1%/yr** while its compounded outcome is the *opposite sign* (-7.71% vs +2.58% CAGR). A 97%-vol crash-and-rebound inflates the arithmetic mean by roughly $\sigma^2/2$; the geometric investor never saw it.

> 💡 *In plain words:* the one number that makes the premium look real is an artefact of averaging daily percentage moves on something that halved and doubled.

In [4]:
print(f"wins {R['wins']}/{R['n_pairs_win']}  Wilson [{R['wilson_lo']:.2f}, {R['wilson_hi']:.2f}]")
print(f"median pair spread {R['median_pair']:+.2f}%/yr   vs panel mean {R['pairwise']:+.2f}%/yr")
print(f"drop B. Riley -> pairwise {R['ex_rily']:+.2f}%/yr (t = {R['t_ex_rily']:+.2f})")
print(f"drop B. Riley + B&W -> pairwise {R['ex2_pw']:+.2f}%/yr (t = {R['ex2_t']:+.2f})  "
      f"adv {R['ex2_adv']:+.3f}  -> SIGN FLIP on 6 remaining pairs")

wins 6/8  Wilson [0.41, 0.93]
median pair spread +0.70%/yr   vs panel mean +3.10%/yr
drop B. Riley -> pairwise +0.81%/yr (t = +0.17)
drop B. Riley + B&W -> pairwise -0.60%/yr (t = -0.21)  adv -0.116  -> SIGN FLIP on 6 remaining pairs


## 3b. The risk ordering is an artefact too

The tempting fallback — *fine, the premium is unmeasurable, but at least the junior rung is reliably riskier* — does not survive two checks.

**Frequency.** These are thin retail tapes: mean stale-print share **5.4% / 5.0%** (pref / bond), with CMS-PB at **15.5%** and lag-1 AC **-0.28** — textbook bid-ask bounce, which *inflates* measured daily variance. Compounding the same returns up:

| frequency | vol ratio | pref riskier |
|---|--:|--:|
| daily | 1.31 | **6/8** |
| weekly | 1.30 | 4/8 |
| monthly | 1.30 | **2/8** |

CMS-PB alone goes from **24.0%** daily vol to **11.4%** monthly — below its own bond's **13.5%**.

**Concentration.** The panel ratio holds up only because two names dominate an equal-weight variance. Drop B. Riley and Babcock & Wilcox:

| panel | pairwise (*t*) | adv | ratio d / m | pref riskier (m) | DD pref vs bond |
|---|--:|--:|--:|--:|--:|
| all 8 | +3.10% (+0.55) | +0.166 | 1.31 / 1.30 | 2/8 | -30.8% vs -20.4% |
| ex B. Riley | +0.81% (+0.17) | +0.176 | 1.30 / 1.27 | 1/7 | −23.1% vs −19.4% |
| **ex both** | **-0.60%** (-0.21) | **-0.116** | **0.94 / 0.79** | **0/6** | -20.1% vs -19.3% |

> 💡 *In plain words:* the junior rung is the wilder one **when the issuer is in distress** — a tautology about distress, not a price of subordination. In the 6 non-distressed pairs there is no step in either direction, and it is worth noting those are also the pairs whose seniority gap is narrowest (`data.RUNG_STRUCTURE`: only 2 of 8 pairs put a *senior* note against a perpetual preferred).

In [5]:
for f, ratio, riskier in (("daily", R['ratio_d'], R['riskier_d']),
                          ("weekly", R['ratio_w'], R['riskier_w']),
                          ("monthly", R['ratio_m'], R['riskier_m'])):
    print(f"{f:8s} vol ratio {ratio:.2f}   pref riskier in {riskier}/8 pairs")
print()
print(f"ex B.Riley + B&W: ratio {R['ex2_ratio_d']:.2f} daily / {R['ex2_ratio_m']:.2f} monthly, "
      f"pref riskier {R['ex2_riskier_m']}/{R['ex2_pairs']}, "
      f"DD {R['ex2_dd_pref']:.1f}% vs {R['ex2_dd_bond']:.1f}%")
print('-> neither the return step nor the risk step is a seniority fact')

daily    vol ratio 1.31   pref riskier in 6/8 pairs
weekly   vol ratio 1.30   pref riskier in 4/8 pairs
monthly  vol ratio 1.30   pref riskier in 2/8 pairs

ex B.Riley + B&W: ratio 0.94 daily / 0.79 monthly, pref riskier 0/6, DD -20.1% vs -19.3%
-> neither the return step nor the risk step is a seniority fact


## 4. Era cut, cost sweep, borrow sweep

The spread changes sign across the halves; cost is irrelevant (it cancels between two near-identical-turnover ladders); the borrow assumption is what actually kills the dollar-neutral expression.

In [6]:
print(f"early n={R['era_e_n']}: adv {R['era_e_adv']:+.2f}  pairwise {R['era_e_pw']:+.2f}% (t={R['era_e_t']:+.2f})")
print(f"late  n={R['era_l_n']}: adv {R['era_l_adv']:+.2f}  pairwise {R['era_l_pw']:+.2f}% (t={R['era_l_t']:+.2f})  <- sign flip")
print()
print(f"cost   5 bps: adv {R['cost5_adv']:+.3f}  spread {R['cost5_spread']:+.2f}%")
print(f"cost 100 bps: adv {R['cost100_adv']:+.3f}  spread {R['cost100_spread']:+.2f}%  -> cost is not the story")
print()
print(f"long-short, borrow   0 bp: {R['ls_gross']:+.2f}%/yr  Sharpe {R['ls_sharpe0']:.3f}  t {R['ls_t']:+.2f}")
print(f"long-short, borrow 300 bp: {R['ls_300']:+.2f}%/yr  Sharpe {R['ls_sharpe300']:.3f}  DD {R['ls_dd']:.1f}%")

early n=542: adv -0.16  pairwise -1.16% (t=-0.21)
late  n=564: adv +0.10  pairwise +7.17% (t=+0.72)  <- sign flip

cost   5 bps: adv +0.166  spread +4.33%
cost 100 bps: adv +0.167  spread +4.14%  -> cost is not the story

long-short, borrow   0 bp: +4.29%/yr  Sharpe 0.282  t +0.65
long-short, borrow 300 bp: +1.29%/yr  Sharpe 0.085  DD -25.9%


## 5. Stress windows, long history, and what the screen deletes

Seniority is meant to be worth something precisely when the issuer's credit is questioned. It wasn't, in four of five windows — because between two rungs of one issuer the dominant difference is usually **duration**, not subordination.

In [7]:
print(f"2022 rate shock   gap {R['st_2022']:+.2f} pp   (a dead heat)")
print(f"Mar-2023 banks    gap {R['st_banks']:+.2f} pp   (junior rung won)")
print(f"2024 issuer strss gap {R['st_2024']:+.2f} pp   (junior rung won)")
print(f"2025 tariff shock gap {R['st_tariff']:+.2f} pp   (senior rung won, barely)")
print(f"COVID (2-pair)    gap {R['st_covid']:+.2f} pp   (junior rung won by 9.5 pp)")
print()
print(f"long-history CMS+Duke, n={R['long_n']}: pairwise {R['long_pw']:+.2f}%/yr (t={R['long_t']:+.2f})  "
      f"adv {R['long_adv']:+.3f}  vol ratio {R['long_volratio']:.2f}")
print(f"alternative rung (CMSA/BEPI/BIPI): pairwise {R['alt_pw']:+.2f}%/yr (t={R['alt_t']:+.2f})  "
      f"adv {R['alt_adv']:+.3f}  -> not a CUSIP artefact")
print()
print(f"SURVIVORSHIP  Sachem SACH-PA vs SACC (bond matured 2024-12):")
print(f"   pref cum {R['sach_pref']:+.1f}%  bond cum {R['sach_bond']:+.1f}%  "
      f"spread {R['sach_spread']:+.2f}%/yr (t={R['sach_t']:+.2f})  -> the screen deletes the losers")
print(f"FALLBACK      PFF exSharpe {R['pff_sharpe']:+.3f} (CAGR {R['pff_cagr']:+.2f}%);  "
      f"PFF - baby bonds {R['pff_gap']:+.2f}%/yr (t={R['pff_t']:+.2f})")

2022 rate shock   gap +0.06 pp   (a dead heat)
Mar-2023 banks    gap +2.89 pp   (junior rung won)
2024 issuer strss gap +5.28 pp   (junior rung won)
2025 tariff shock gap -1.03 pp   (senior rung won, barely)
COVID (2-pair)    gap +9.53 pp   (junior rung won by 9.5 pp)

long-history CMS+Duke, n=1821: pairwise -0.43%/yr (t=-0.14)  adv -0.064  vol ratio 0.95
alternative rung (CMSA/BEPI/BIPI): pairwise +3.84%/yr (t=+0.68)  adv +0.222  -> not a CUSIP artefact

SURVIVORSHIP  Sachem SACH-PA vs SACC (bond matured 2024-12):
   pref cum -17.4%  bond cum +21.6%  spread -10.63%/yr (t=-0.85)  -> the screen deletes the losers
FALLBACK      PFF exSharpe -0.191 (CAGR -2.60%);  PFF - baby bonds -4.75%/yr (t=-0.90)


## 6. Live synthetic control — power and calibration

Planted world: eight issuers, junior rung paid a real +6.0%/yr carry on top of its higher factor loading. Null world: identical betas, premium exactly zero. The estimator must fire on the first and stay silent on the second. This proves the machinery; it never supports the real-tape stamp.

In [8]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
import numpy as np
from two_ladders import data, strategy as st
for ss, tag in ((1.0, 'planted +6.0%/yr'), (0.0, 'exact null    ')):
    sp, ts = [], []
    for k in range(8):
        d = st.synthetic_detect(*data.synthetic_panel(signal_strength=ss, seed=933+k)[:3])
        sp.append(d['pairwise_spread_ann']); ts.append(d['t_pairwise'])
    sp, ts = np.array(sp), np.array(ts)
    print(f"{tag}: spread {sp.mean()*100:+.2f}%/yr (sd {sp.std(ddof=1)*100:.2f}%)  "
          f"|t|>=2 in {int((abs(ts)>=2).sum())}/8")

planted +6.0%/yr: spread +5.72%/yr (sd 0.98%)  |t|>=2 in 8/8


exact null    : spread -0.28%/yr (sd 0.98%)  |t|>=2 in 0/8


(The frozen 20-seed run in `docs/results.md`: planted **+5.74%/yr**, fires **20/20**; null **-0.26%/yr**, fires **0/20**.)

## Verdict

- **Signal — None.** Pairwise same-issuer spread **+3.10%/yr**, HAC *t* = **+0.55**, bootstrap CI **[-6.80%, +13.57%]**; Sharpe advantage **+0.166**, CI **[-0.47, +0.80]** with 32% of draws negative. Sign flips by era (-1.16% / +7.17%); collapses to **+0.81% (t=+0.17)** ex-B. Riley; median pair **+0.70%**; hit rate 6/8 with a Wilson interval spanning a coin flip; the seven-year two-pair extension is **-0.43%/yr**; the survivorship probe runs **-10.63%/yr** the *other* way; and PFF **lost 4.75%/yr** to the baby-bond basket on the same window. The synthetic control recovers a planted +6%/yr on 20/20 seeds and is silent on 0/20 nulls, so the estimator has the power the tape lacks. And the risk ordering that once looked robust (1.42x vol, riskier in 6/8 pairs) is **daily microstructure plus two distressed names**: 2/8 at monthly frequency, and 0/6 once those two names leave, where the ratio itself is 0.79.
- **Tradability — Mirage.** Dollar-neutral long-pref / short-bond: **+4.29%/yr** gross, Sharpe 0.282, *t* +0.65, DD -25.9%; Sharpe 0.085 at a 300 bp borrow ASSUMPTION. Cost is irrelevant (adv +0.166 → +0.167 from 5 to 100 bps) — the signal is what is missing. The directional tilt is no better: the long-history extension pays the junior rung **less**, and the live-listing screen deletes the pairs that would drag the panel down.
- **Named limits.** 4.4-year common window, one rate cycle, no default cycle, 8 correlated pairs; survivorship on both rungs; a 25 bps cost PROXY and a borrow ASSUMPTION, both swept; thin tapes (stale-print shares to 15.5%, lag-1 AC to -0.28), which is why every risk claim is stated at three frequencies; and a seniority step that is only the textbook one in 2 of 8 pairs (`data.RUNG_STRUCTURE`) — a limit of the *listed* universe, not a fixable choice.